# External Dependencies for SCIP
<br>  

Language-agnostic analysis of external dependencies detected from SCIP index data.
Groups by external artifact (`module` property) across all languages.

### References
- [jqassistant](https://jqassistant.org)
- [Neo4j Python Driver](https://neo4j.com/docs/api/python-driver/current)
- [SCIP Index Format](https://sourcegraph.com/blog/announcing-scip)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plot
from neo4j import GraphDatabase
from typing import Any, cast, LiteralString

In [ ]:
# Please set the environment variable "NEO4J_INITIAL_PASSWORD" in your shell
# before starting jupyter notebook to provide the password for the user "neo4j".
# It is not recommended to hardcode the password into jupyter notebook for security reasons.

neo4j_password = os.environ.get("NEO4J_INITIAL_PASSWORD")
if not neo4j_password:
    raise ValueError("NEO4J_INITIAL_PASSWORD environment variable must be set")

driver = GraphDatabase.driver(uri="bolt://localhost:7687", auth=("neo4j", neo4j_password))
driver.verify_connectivity()

In [ ]:
def get_cypher_query_from_file(filename: str) -> str:
    with open(filename, encoding="utf-8") as file:
        return " ".join(file.readlines())


def query_cypher_to_data_frame(filename: str) -> pd.DataFrame:
    records, _, keys = driver.execute_query(cast(LiteralString, get_cypher_query_from_file(filename)))
    return pd.DataFrame([r.values() for r in records], columns=keys)


main_color_map: str = "nipy_spectral"

In [ ]:
def group_to_others_below_threshold(
    data_frame: pd.DataFrame, value_column: str, name_column: str, threshold: float
) -> pd.DataFrame:
    """
    Add percentage column and groups values below threshold to 'others'.

    Args:
        data_frame: Input pandas DataFrame
        value_column: Name of the column that contains the numeric value
        name_column: Name of the column that contains the group name
        threshold: Threshold in % to group values below it into 'others'

    Returns:
        DataFrame with grouped values sorted descending by percentage
    """
    result = data_frame[[name_column, value_column]].copy()
    percent_column = value_column + "Percent"
    result[percent_column] = result[value_column] / result[value_column].sum() * 100.0
    result[name_column] = result[name_column].astype(str)
    result.loc[result[percent_column] < threshold, name_column] = "others"
    result = result.groupby(name_column).sum()
    return result.sort_values(by=percent_column, ascending=False)

In [ ]:
def filter_values_below_threshold(
    data_frame: pd.DataFrame, value_column: str, upper_limit: float = 100.0
) -> pd.DataFrame:
    """
    Add percentage column and filter entries.

    Args:
        data_frame: Input pandas DataFrame
        value_column: Name of the column that contains the numeric value
        upper_limit: Defaults to 100%. Filters out all entries exceeding this limit.

    Returns:
        Filtered DataFrame sorted descending by percentage
    """
    result = data_frame.copy()
    percent_column = value_column + "Percent"
    result[percent_column] = result[value_column] / result[value_column].sum() * 100.0
    result = result.query(f"`{percent_column}` <= {upper_limit}")
    result = result.reset_index(drop=True)
    return result.sort_values(by=percent_column, ascending=False)

In [ ]:
import numpy as np


def explode_index_value(
    input_data_frame: pd.DataFrame,
    index_value_to_emphasize: str = "others",
    base_value: float = 0.02,
    emphasize_value: float = 0.2,
) -> Any:
    """
    Generate explode offsets for pie chart slices.

    Args:
        input_data_frame: Input pandas DataFrame with data to plot
        index_value_to_emphasize: Value of the index to emphasize (Default='others')
        base_value: Base offset value for all slices (Default=0.02)
        emphasize_value: Offset value for the emphasized slice (Default=0.2)

    Returns:
        Array with explode offset for each slice
    """
    return (input_data_frame.index == index_value_to_emphasize) * emphasize_value + base_value

In [ ]:
def plot_pie_chart(input_data_frame: pd.DataFrame, title: str) -> None:
    """Render and display a pie chart from a DataFrame."""
    if input_data_frame.empty:
        print(f"No data to plot for title '{title}'.")
        return

    value_column = input_data_frame.columns[0]
    total_sum = input_data_frame[value_column].sum()

    def custom_auto_percentage_format(percentage: float) -> str:
        return f"{percentage:1.2f}% ({total_sum * percentage / 100.0:.0f})"

    plot.figure()
    axis = input_data_frame.plot(
        kind="pie",
        y=value_column + "Percent",
        ylabel="",
        legend=True,
        labeldistance=None,
        autopct=custom_auto_percentage_format,
        textprops={"fontsize": 6},
        pctdistance=1.15,
        cmap=main_color_map,
        figsize=(9, 9),
        explode=explode_index_value(input_data_frame, index_value_to_emphasize="others"),
    )
    plot.title(title, pad=15)
    axis.legend(bbox_to_anchor=(1.08, 1), loc="upper left")
    plot.show()

In [ ]:
%%html
<style>
/* CSS style for smaller dataframe tables. */
.dataframe th {
    font-size: 8px;
}
.dataframe td {
    font-size: 8px;
}
</style>

## 1. SCIP External Artifact Usage Overall

Groups all internal-type-to-external-type `DEPENDS_ON` edges by the external artifact (`module` property).
Language-agnostic: all languages in the SCIP index are combined.

Source query: `External_artifact_usage_overall_for_Scip.cypher`

**Columns:**
- *externalArtifactName* — the `module` property of the external type (external artifact identifier)
- *numberOfInternalCallerModules* — distinct internal modules with at least one type depending on this external artifact
- *numberOfInternalCallerTypes* — distinct internal types depending on this external artifact
- *numberOfTypeCalls* — total number of `DEPENDS_ON` edges
- *totalReferenceCount* — sum of `referenceCount` properties on `DEPENDS_ON` edges
- *allInternalModules* — sample of internal module names (up to 10)
- *allInternalTypes* — sample of internal type fully qualified names (up to 10)
- *tenExternalTypeNames* — sample of external type names in this artifact (up to 10)

In [ ]:
artifact_usage_overall = query_cypher_to_data_frame("../queries/External_artifact_usage_overall_for_Scip.cypher")
artifact_usage_overall.head(20)

### Chart 1a — Top external artifacts by internal caller types (≥0.7%)

In [ ]:
plot_pie_chart(
    input_data_frame=group_to_others_below_threshold(
        data_frame=artifact_usage_overall,
        value_column="numberOfInternalCallerTypes",
        name_column="externalArtifactName",
        threshold=0.7,
    ),
    title="Top external artifacts by internal caller types (≥0.7%)",
)

### Chart 1b — Top external artifacts by internal caller modules (≥0.7%)

In [ ]:
plot_pie_chart(
    input_data_frame=group_to_others_below_threshold(
        data_frame=artifact_usage_overall,
        value_column="numberOfInternalCallerModules",
        name_column="externalArtifactName",
        threshold=0.7,
    ),
    title="Top external artifacts by internal caller modules (≥0.7%)",
)

## 2. SCIP External Artifact Usage Spread

For each external artifact: how many distinct internal artifacts use it.
High spread indicates a pervasive cross-cutting dependency.

Source query: `External_artifact_usage_spread_for_Scip.cypher`

**Columns:**
- *externalArtifactName* — the external artifact identifier
- *numberOfInternalArtifacts* — how many distinct internal artifacts depend on this external artifact
- *sumNumberOfModules* — total internal modules using this external artifact across all internal artifacts
- *sumNumberOfTypes* — total internal types using this external artifact across all internal artifacts
- *sumReferenceCount* — total reference count across all dependencies

In [ ]:
artifact_usage_spread = query_cypher_to_data_frame("../queries/External_artifact_usage_spread_for_Scip.cypher")
artifact_usage_spread.head(20)

### Chart 2a — Most spread external artifacts by types (≥0.5%)

In [ ]:
plot_pie_chart(
    input_data_frame=group_to_others_below_threshold(
        data_frame=artifact_usage_spread,
        value_column="sumNumberOfTypes",
        name_column="externalArtifactName",
        threshold=0.5,
    ),
    title="Most spread external artifacts by total internal types (≥0.5%)",
)

## 3. SCIP External Artifact Usage per Internal Artifact

Per internal artifact and external artifact: module and type counts.
Useful for understanding which internal components have the most external coupling.

Source query: `External_artifact_usage_per_internal_artifact_for_Scip.cypher`

**Columns:**
- *internalArtifactName* — the internal artifact (project/module group)
- *externalArtifactName* — the external artifact depended upon
- *numberOfModules* — internal modules in this artifact depending on this external artifact
- *numberOfTypes* — internal types in this artifact depending on this external artifact
- *totalReferenceCount* — sum of reference counts

In [ ]:
artifact_usage_per_artifact = query_cypher_to_data_frame(
    "../queries/External_artifact_usage_per_internal_artifact_for_Scip.cypher"
)
artifact_usage_per_artifact.head(30)

## 4. SCIP External Artifact Usage per Internal Module

Internal modules ranked by number of distinct external artifacts they depend on.
Modules with high external artifact counts are candidates for refactoring.

Source query: `External_artifact_usage_per_internal_module_sorted_for_Scip.cypher`

**Columns:**
- *internalModuleName* — the internal module
- *internalArtifactName* — the artifact containing the module
- *numberOfExternalArtifacts* — distinct external artifacts depended on by this module
- *numberOfExternalTypes* — distinct external types depended on
- *totalReferenceCount* — sum of reference counts

In [ ]:
module_usage_sorted = query_cypher_to_data_frame(
    "../queries/External_artifact_usage_per_internal_module_sorted_for_Scip.cypher"
)
module_usage_sorted.head(30)